In [19]:
import vectice as vct

vec_project = vct.connect(config="../../.config/token_i.json")

Welcome, 'Ines Cerdan'. You`re now successfully connected to the project '_FFBank & LuxAir Co-Branded Credit Card Initial offer' in Vectice.

To access a specific phase, use project.phase(Phase ID)
To get a list of phases you can access and their IDs, use project.list_phases()

For quick access to the list of phases in the Vectice web app, visit:
https://app.vectice.com/browse/project/PRJ-189


### Read datasets:
 - PTY_ID_MAIN - From our BigQuery Dev environment
 - HIST_TRANS - From our BigQuery Dev environment
 - LuxAir_Accts - S3
 - OFAC_SDN - S3

In [20]:
# Connect to BigQuery Dev using Service Account
from google.cloud import bigquery
from google.oauth2 import service_account
creds = service_account.Credentials.from_service_account_file("../../.config/bq_dev_sa.json", scopes=["https://www.googleapis.com/auth/cloud-platform"])

bigquery_client = bigquery.Client(
    credentials= creds,
    project=creds.project_id
)

Query PTY_ID_MAIN and HIST_TRX from our Dev BigQuery env. Retrieving full tables, we will need to remove non US customers from the resultsets as per compliance.

In [21]:
# Query PTY_ID_MAIN table
qry_PTY_ID_MAIN = "SELECT * FROM `solutions-engineering-363108.CUST_PTY_INFO.PTY_ID_MAIN`"
#Run the query and write result to a pandas data frame
Query_Results = bigquery_client.query(qry_PTY_ID_MAIN)
df_PTY_ID_MAIN = Query_Results.to_dataframe()
#View top few rows of result
df_PTY_ID_MAIN.head()

,Customer_PTY_Id,Customer_Surname,Customer_Given_Name,Customer_Email,Customer_Gender,Customer_DOB,Customer_POB,Customer_Street_Address,Customer_State_Address,Customer_Zip_Address,Customer_Income,Customer_SSN,Customer_ITIN,Customer_Employer,Customer_Account_Open_Date,Customer_Initial_Deposit,Customer_Address_Cntry
0,65-3683456,Ney,Alf,aney36@wikipedia.org,Male,1999-02-14,Brea,6760 Corry Hill,CA,92822,229399.52,178-89-2425,576-23-1520,Muxo,1996-07-28,7736.97,US
1,06-0539217,Corcoran,Kev,kcorcoran1p@wp.com,Male,1987-02-05,Brea,2 Dryden Terrace,CA,92822,1109648.60,752-91-3542,646-53-1152,Kare,2005-12-12,60650.37,US
2,33-8558058,Doxsey,Julian,jdoxsey8l@mit.edu,Male,1964-03-01,Brea,30 Stephen Crossing,CA,92822,254149.73,636-48-4129,160-12-9075,Tambee,2004-03-26,14382.47,US
3,19-2047408,Thies,Grady,gthies2q@addthis.com,Male,2002-10-26,Erie,6414 Harper Junction,PA,16550,1079876.08,835-02-3608,295-66-1574,Yabox,2008-01-22,2823.16,US
4,08-0960611,O' Meara,Desi,domearaqd@bing.com,Male,1928-03-12,Erie,3 Melby Hill,PA,16550,1203616.91,567-95-1426,685-08-6154,Divape,1997-12-16,9901.73,US


In [22]:
# Query HIST_TRANS table
qry_HIST_TRX = "SELECT * FROM `solutions-engineering-363108.HIST_CUST_INFO.HIST_TRANS`"
#Run the query and write result to a pandas data frame
Query_Results = bigquery_client.query(qry_HIST_TRX)
df_HIST_TRX = Query_Results.to_dataframe()
#View top few rows of result
df_HIST_TRX.head()

,TransactionID,Customer_PTY_ID,AccountNumber,Email,Gender,Amount,Date,TransactionType
0,768,97-0539166,GT46 XFXO VIXP SUMO O1AE WXIJ K6G7,gismead2x@google.ru,Male,515287.07,2017-03-14,Wire
1,768,91-4224986,TN58 1132 2103 0923 2635 4596,helcoux4b@ucla.edu,Male,113210.26,2017-08-14,Wire
2,768,55-6039702,RS97 6933 0668 4971 9957 29,adimitruea@rambler.ru,Female,769819.63,2017-12-17,Wire
3,768,04-6258749,AD12 7772 3942 OKBX BHWK MPY2,mmunningsh3@hugedomains.com,Female,48354.43,2018-02-01,Wire
4,768,82-4918676,PL37 5346 9955 0710 9093 8989 3441,ldengefv@digg.com,Male,237134.26,2018-11-03,Wire


Reading the two external files from our S3 bucket.

In [23]:
# Read the external files from S3
# Create connection
from boto3 import client
from botocore import UNSIGNED
from botocore.client import Config
import s3fs

s3_client = client('s3', config=Config(signature_version=UNSIGNED), region_name='us-west-1')


In [24]:
import pandas as pd
# Read the external files in dataframes
s3 = s3fs.S3FileSystem(anon=True)

with s3.open("vectice-examples/Samples Data/LuxAir_Accts.csv", mode="rb") as f:
    df_LuxAir_Accts = pd.read_csv(f)

with s3.open("vectice-examples/Samples Data/OFAC_SDN.csv", mode="rb") as f:
    df_OFAC_SDN = pd.read_csv(f)

Document my findings in Vectice

In [25]:
from vectice import Dataset, S3Resource
from vectice.models.resource import BigQueryResource

iteration = vec_project.phase("Data Collection").create_or_get_current_iteration()

Phase 'Data Collection' successfully retrieved.

For quick access to the Phase in the Vectice web app, visit:
https://app.vectice.com/browse/phase/PHA-1075
Iteration 'Iteration 7' successfully retrieved.

For quick access to the Iteration in the Vectice web app, visit:
https://app.vectice.com/browse/iteration/ITR-11187


In [26]:
vct_PTY_ID_MAIN = BigQueryResource (bq_client=bigquery_client, paths="solutions-engineering-363108.CUST_PTY_INFO.PTY_ID_MAIN", dataframes = df_PTY_ID_MAIN)
vct_HIST_TRX = BigQueryResource (bq_client=bigquery_client, paths="solutions-engineering-363108.HIST_CUST_INFO.HIST_TRANS", dataframes = df_HIST_TRX)

vct_LuxAir_Accts = S3Resource(uris="s3://vectice-examples/Samples Data/LuxAir_Accts.csv", dataframes = df_LuxAir_Accts)
vct_OFAC_SDN = S3Resource(uris="s3://vectice-examples/Samples Data/OFAC_SDN.csv", dataframes = df_OFAC_SDN)

In [27]:


# Documenting all four datasets used in the project
iteration.log(Dataset.origin(name="PTY_ID_MAIN", resource=vct_PTY_ID_MAIN, properties={"SQL":qry_PTY_ID_MAIN}, attachments="PTY_ID_MAIN_boxplot.jpg"))
iteration.log(Dataset.origin(name="HIST_TRANSACTIONS", resource=vct_HIST_TRX, properties={"SQL":qry_HIST_TRX}, attachments="HIST_TRX_histogram.jpg"))
iteration.log(Dataset.origin(name="LuxAir_Accts", resource=vct_LuxAir_Accts))
iteration.log(Dataset.origin(name="OFAC_SDN", resource=vct_OFAC_SDN))

iteration.log("We have identified the proper datasets for this project. \nTwo of the datasets (\"LuxAir_Accts\" and \"OFAC_SDN\") are coming from external sources and are dropped weekly on our S3 bucket. These files will need to be automated.")

Existing dataset: 'PTY_ID_MAIN' and version: 'Version 4' already linked to iteration: 'Iteration 7'.
Attachments: PTY_ID_MAIN_boxplot.jpg
Link to iteration: https://app.vectice.com/browse/iteration/ITR-11187

Existing dataset: 'HIST_TRANSACTIONS' and version: 'Version 4' already linked to iteration: 'Iteration 7'.
Attachments: HIST_TRX_histogram.jpg
Link to iteration: https://app.vectice.com/browse/iteration/ITR-11187

Existing dataset: 'LuxAir_Accts' and version: 'Version 5' already linked to iteration: 'Iteration 7'.
Attachments: None
Link to iteration: https://app.vectice.com/browse/iteration/ITR-11187

Existing dataset: 'OFAC_SDN' and version: 'Version 5' already linked to iteration: 'Iteration 7'.
Attachments: None
Link to iteration: https://app.vectice.com/browse/iteration/ITR-11187

Added comment to iteration: 'Iteration 7'.
Link to iteration: https://app.vectice.com/browse/iteration/ITR-11187



Capture data summary - Describe data, check for N/A, etc...

In [ ]:
df_PTY_ID_MAIN.describe()

In [ ]:
df_PTY_ID_MAIN.shape[0]


In [ ]:
df_PTY_ID_MAIN.shape[1]

In [ ]:
df_PTY_ID_MAIN.isnull().sum().sum()

In [ ]:
df_HIST_TRX.describe()

In [ ]:
df_HIST_TRX.shape[0]

In [ ]:
df_HIST_TRX.shape[1]

In [ ]:
df_HIST_TRX.isnull().sum().sum()

In [ ]:
# Log insights in Vectice
msg = "\nSize of Original Dataset:\n"\
"PTY_ID_MAIN: Observations: " + str(df_PTY_ID_MAIN.shape[0]) + " - Features: " + str(df_PTY_ID_MAIN.shape[1])  + "- # of null values: " + str(df_PTY_ID_MAIN.isnull().sum().sum()) + "\n" \
"HIST_TRX: Observations: " + str(df_HIST_TRX.shape[0])  + "- Features: " + str(df_HIST_TRX.shape[1]) + "- # of null values: " + str(df_HIST_TRX.isnull().sum().sum()) + "\n" \
"LuxAir_Accts: Observations: " + str(df_LuxAir_Accts.shape[0])  + " - Features: " + str(df_LuxAir_Accts.shape[1]) + "- # of null values: " + str(df_LuxAir_Accts.isnull().sum().sum()) + "\n" \
"OFAC_SDN: Observations: " + str(df_OFAC_SDN.shape[0])  + " - Features: " + str(df_OFAC_SDN.shape[1]) + "- # of null values: " + str(df_OFAC_SDN.isnull().sum().sum())

iteration.log("The data properties have been reviewed for the datasets identified\n" + msg)

Visualize the data

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sb

df_PTY_ID_MAIN.plot(kind='box', subplots=True, layout=(2,2), sharex=False, sharey=False)
plt.savefig("PTY_ID_MAIN_boxplot.jpg")
plt.show()

df_PTY_ID_MAIN.hist()
histogram = plt.savefig("PTY_ID_MAIN_histogram.jpg")
plt.show()

In [ ]:
print("HIST_TRX Visualizations:")

df_HIST_TRX.plot(kind='box', subplots=True, layout=(2,2), sharex=False, sharey=False)
plt.savefig("HIST_TRX_boxplot.jpg")
plt.show()
df_HIST_TRX.hist()
histogram = plt.savefig("HIST_TRX_histogram.jpg")
plt.show()

In [ ]:
print("LuxAir_Accts Visualizations:")

df_LuxAir_Accts.plot(kind='box', subplots=True, layout=(2,2), sharex=False, sharey=False)
plt.savefig("LuxAir_Accts_boxplot.jpg")
plt.show()
df_LuxAir_Accts.hist()
histogram = plt.savefig("LuxAir_Accts_histogram.jpg")
plt.show()

In [ ]:
# Capture the visualizations in Vectice

iteration.log("PTY_ID_MAIN_plot.jpg")
iteration.log("PTY_ID_MAIN_boxplot.jpg")
iteration.log("PTY_ID_MAIN_histogram.jpg")

#iteration.step_explore_data += "HIST_TRX_plot.jpg"
iteration.log("HIST_TRX_boxplot.jpg")
iteration.log("HIST_TRX_histogram.jpg")

#iteration.step_explore_data += "LuxAir_Accts_plot.jpg"
iteration.log("LuxAir_Accts_boxplot.jpg")
iteration.log("LuxAir_Accts_histogram.jpg")



In [ ]:
iteration.complete()